In [ ]:
"""
FM Student Training on teacher pairs
"""
import copy
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
from torch.optim import AdamW
from tqdm import tqdm

from ml_conformer_generator.src.mlconfgen.egnn import EGNNDynamics
from ml_conformer_generator.src.mlconfgen.equivariant_flow_matching import EquivariantFlowMatching
from ml_conformer_generator.src.mlconfgen.utils import CONTEXT_NORMS, MAX_N_NODES
from ml_conformer_generator.src.mlconfgen.utils.mol_utils import prepare_masks

# --------------------- config ---------------------
device = "cuda"
BATCH = 512
EPOCHS = 1001
CKPT_EVERY = 5
LR = 1e-4              # EDM train default; try 1e-5 if unstable
HIDDEN_NF = 128
PAD_TO = MAX_N_NODES
EMA_DECAY = 0.999
EARLY_STOP_PATIENCE = 5   # epochs with < MIN_DELTA improvement
MIN_DELTA = 0.01
PART_CYCLE = [0, 1]       # or [0, 1, 2, 3, 2, 1]

PAIR_DIR = Path("./teacher_pairs/teacher_pairs")
EDM_WEIGHTS = "edm_moi_chembl_15_39.pt"
CKPT_DIR = Path("./checkpoints_fm")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = CKPT_DIR / f"train_{HIDDEN_NF}.log"


# --------------------- EDM-style helpers ---------------------
class EMA:
    def __init__(self, beta: float):
        self.beta = beta

    def update_model_average(self, ma_model, current_model):
        for cur, ma in zip(current_model.parameters(), ma_model.parameters()):
            ma.data = self.update_average(ma.data, cur.data)

    def update_average(self, old, new):
        if old is None:
            return new
        return old * self.beta + (1.0 - self.beta) * new


class Queue:
    def __init__(self, max_len: int = 50):
        self.items = []
        self.max_len = max_len

    def __len__(self):
        return len(self.items)

    def add(self, item):
        self.items.insert(0, item)
        if len(self) > self.max_len:
            self.items.pop()

    def mean(self):
        return float(np.mean(self.items))

    def std(self):
        return float(np.std(self.items))


def gradient_clipping(model, gradnorm_queue: Queue):
    max_grad_norm = 1.5 * gradnorm_queue.mean() + 2.0 * gradnorm_queue.std()
    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(), max_norm=max_grad_norm, norm_type=2.0
    )
    gn = float(grad_norm)
    if gn > max_grad_norm:
        gradnorm_queue.add(float(max_grad_norm))
        print(f"Clipped gradient {gn:.1f} (allowed {max_grad_norm:.1f})")
    else:
        gradnorm_queue.add(gn)


def log(msg: str) -> None:
    line = f"{datetime.now().isoformat(timespec='seconds')}  {msg}"
    print(line)
    with open(LOG_PATH, "a") as f:
        f.write(line + "\n")


def load_part_pairs(part_i: int):
    paths = sorted(PAIR_DIR.glob(f"part_{part_i + 1}_shard*.pt"))
    if not paths:
        # also try part{N}_shard / flat shards
        paths = sorted(PAIR_DIR.glob(f"part{part_i + 1}_shard*.pt"))
    if not paths:
        paths = sorted(PAIR_DIR.glob("shard*.pt"))
        if part_i != 0:
            raise FileNotFoundError(f"no part shards; only flat shards in {PAIR_DIR}")
    if not paths:
        raise FileNotFoundError(f"no shards for part {part_i + 1} in {PAIR_DIR}")
    packs = [torch.load(p, map_location="cpu", weights_only=False) for p in paths]
    return (
        torch.cat([p["z_T"] for p in packs]),
        torch.cat([p["x1"] for p in packs]),
        torch.cat([p["n_atoms"] for p in packs]),
        torch.cat([p["context"] for p in packs]),
    )


def save_ckpt(path, epoch, avg, best, student, student_ema, opt, gradnorm_queue):
    torch.save(
        {
            "epoch": epoch,
            "avg_loss": avg,
            "best_loss": best,
            "student": student.state_dict(),
            "student_ema": student_ema.state_dict() if student_ema is not None else None,
            "opt": opt.state_dict(),
            "hidden_nf": HIDDEN_NF,
            "lr": LR,
            "batch": BATCH,
            "ema_decay": EMA_DECAY,
            "gradnorm_queue": list(gradnorm_queue.items),
        },
        path,
    )


# --------------------- model / optim ---------------------
edm_ckpt = torch.load(EDM_WEIGHTS, map_location="cpu", weights_only=False)
norms = {
    k: torch.tensor(v, device=device, dtype=torch.float32)
    for k, v in edm_ckpt.get("context_norms", CONTEXT_NORMS).items()
}

student = EquivariantFlowMatching(
    dynamics=EGNNDynamics(
        in_node_nf=9, context_node_nf=3, hidden_nf=HIDDEN_NF, device=device
    ),
    in_node_nf=8,
).to(device)

student_ema = copy.deepcopy(student).to(device)
for p in student_ema.parameters():
    p.requires_grad_(False)
ema = EMA(EMA_DECAY)

opt = AdamW(
    student.parameters(),
    lr=LR,
    amsgrad=True,
    weight_decay=1e-12,
)

gradnorm_queue = Queue()
gradnorm_queue.add(3000.0)  # same seed value as EDM train

log(
    f"start device={device} batch={BATCH} hidden_nf={HIDDEN_NF} lr={LR} "
    f"ema={EMA_DECAY} epochs={EPOCHS} adaptive_clip=True"
)

# --------------------- train ---------------------
best_loss = float("inf")
epochs_no_improve = 0

for epoch in range(EPOCHS):
    part_i = PART_CYCLE[epoch % len(PART_CYCLE)]
    z_T, x1, n_atoms, context = load_part_pairs(part_i)
    perm = torch.randperm(z_T.shape[0])
    z_T, x1, n_atoms, context = z_T[perm], x1[perm], n_atoms[perm], context[perm]
    log(f"epoch={epoch} part={part_i + 1} n={z_T.shape[0]}")

    student.train()
    running, n_steps = 0.0, 0
    pbar = tqdm(
        range(0, z_T.shape[0] - BATCH + 1, BATCH),
        desc=f"ep{epoch} part{part_i + 1}",
    )
    for i in pbar:
        sl = slice(i, i + BATCH)
        na = n_atoms[sl].to(device=device, dtype=torch.long)
        node_mask, edge_mask = prepare_masks(na, PAD_TO, device)
        ctx = context[sl].to(device=device, dtype=torch.float32)
        normed = (ctx - norms["mean"]) / norms["mad"]
        batch_context = normed.unsqueeze(1).expand(-1, PAD_TO, -1) * node_mask

        x0_b = z_T[sl].to(device=device, dtype=torch.float32)
        x1_b = x1[sl].to(device=device, dtype=torch.float32)

        loss = student.compute_loss(x0_b, x1_b, node_mask, edge_mask, batch_context)
        if not torch.isfinite(loss):
            log(f"WARN non-finite epoch={epoch} step={n_steps}")
            continue

        opt.zero_grad(set_to_none=True)
        loss.backward()
        gradient_clipping(student, gradnorm_queue)
        opt.step()
        ema.update_model_average(student_ema, student)

        running += loss.item()
        n_steps += 1
        if n_steps % 20 == 0:
            pbar.set_postfix(loss=f"{running / n_steps:.4f}")

    avg = running / max(n_steps, 1)
    improved = avg < best_loss - MIN_DELTA
    if avg < best_loss:
        best_loss = avg
        save_ckpt(
            CKPT_DIR / f"best_{HIDDEN_NF}.pt",
            epoch, avg, best_loss, student, student_ema, opt, gradnorm_queue,
        )
        log(f"ckpt best epoch={epoch} avg_loss={avg:.6f}")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    log(
        f"epoch={epoch} part={part_i + 1} steps={n_steps} "
        f"avg_loss={avg:.6f} best_loss={best_loss:.6f} "
        f"grad_q_mean={gradnorm_queue.mean():.1f}"
    )

    if (epoch + 1) % CKPT_EVERY == 0:
        save_ckpt(
            CKPT_DIR / f"latest_{HIDDEN_NF}.pt",
            epoch, avg, best_loss, student, student_ema, opt, gradnorm_queue,
        )
        log(f"ckpt latest epoch={epoch} avg_loss={avg:.6f}")

    if epochs_no_improve >= EARLY_STOP_PATIENCE:
        log(f"early stop at epoch={epoch} best_loss={best_loss:.6f}")
        break

log(f"done best_loss={best_loss:.6f}")

/tmp/ipykernel_41441/3913875974.py:49: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  edm_ckpt = torch.load(EDM_WEIGHTS, map_location="cpu")


2026-09-13T08:58:43  start offline device=cuda batch=128 hidden_nf=128 lr=1e-05


/tmp/ipykernel_41441/3913875974.py:41: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  z_T = torch.cat([torch.load(p, map_location="cpu")["z_T"] for p in paths])
/tmp/ipykerne

2026-09-13T08:58:44  epoch=0 part=1 n=199680


ep0 part1:  30%|██▉       | 462/1560 [02:48<06:40,  2.74it/s, loss=25.8591]

In [ ]:
"""
Offline FM student: 3 linear segments z_T → z_a → z_b → x1
(z_a = teacher reverse after T/3, z_b after 2T/3).
"""
import copy
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
from torch.optim import AdamW
from tqdm import tqdm

from ml_conformer_generator.src.mlconfgen.egnn import EGNNDynamics
from ml_conformer_generator.src.mlconfgen.equivariant_flow_matching import EquivariantFlowMatching
from ml_conformer_generator.src.mlconfgen.utils import CONTEXT_NORMS, MAX_N_NODES, remove_mean_with_mask
from ml_conformer_generator.src.mlconfgen.utils.mol_utils import prepare_masks

# --------------------- config ---------------------
device = "cuda"
BATCH = 128
EPOCHS = 40
CKPT_EVERY = 5
LR = 1e-4
HIDDEN_NF = 420
PAD_TO = MAX_N_NODES
EMA_DECAY = 0.999
EARLY_STOP_PATIENCE = 5
MIN_DELTA = 0.01
N_SEG = 3
PART_CYCLE = [0, 1]

PAIR_DIR = Path("./teacher_pairs/teacher_pairs_3seg")
EDM_WEIGHTS = "edm_moi_chembl_15_39.pt"
CKPT_DIR = Path("./checkpoints_fm")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = CKPT_DIR / f"train_{HIDDEN_NF}_3seg.log"


class EMA:
    def __init__(self, beta: float):
        self.beta = beta

    def update_model_average(self, ma_model, current_model):
        for cur, ma in zip(current_model.parameters(), ma_model.parameters()):
            ma.data = self.update_average(ma.data, cur.data)

    def update_average(self, old, new):
        if old is None:
            return new
        return old * self.beta + (1.0 - self.beta) * new


class Queue:
    def __init__(self, max_len: int = 50):
        self.items = []
        self.max_len = max_len

    def __len__(self):
        return len(self.items)

    def add(self, item):
        self.items.insert(0, item)
        if len(self) > self.max_len:
            self.items.pop()

    def mean(self):
        return float(np.mean(self.items))

    def std(self):
        return float(np.std(self.items))


def gradient_clipping(model, gradnorm_queue: Queue):
    max_grad_norm = 1.5 * gradnorm_queue.mean() + 2.0 * gradnorm_queue.std()
    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(), max_norm=max_grad_norm, norm_type=2.0
    )
    gn = float(grad_norm)
    if gn > max_grad_norm:
        gradnorm_queue.add(float(max_grad_norm))
        print(f"Clipped gradient {gn:.1f} (allowed {max_grad_norm:.1f})")
    else:
        gradnorm_queue.add(gn)


def log(msg: str) -> None:
    line = f"{datetime.now().isoformat(timespec='seconds')}  {msg}"
    print(line)
    with open(LOG_PATH, "a") as f:
        f.write(line + "\n")


def load_part_pairs(part_i: int):
    paths = sorted(PAIR_DIR.glob(f"part_{part_i + 1}_shard*.pt"))
    if not paths:
        paths = sorted(PAIR_DIR.glob(f"part{part_i + 1}_shard*.pt"))
    if not paths:
        paths = sorted(PAIR_DIR.glob("shard*.pt"))
        if part_i != 0:
            raise FileNotFoundError(f"no part shards; only flat shards in {PAIR_DIR}")
    if not paths:
        raise FileNotFoundError(f"no shards in {PAIR_DIR}")
    packs = [torch.load(p, map_location="cpu", weights_only=False) for p in paths]
    for p in packs:
        if "z_a" not in p or "z_b" not in p:
            raise KeyError(f"{PAIR_DIR} missing z_a/z_b — run the dump cell first")
    waypoints = torch.stack(
        [
            torch.cat([p["z_T"] for p in packs]),
            torch.cat([p["z_a"] for p in packs]),
            torch.cat([p["z_b"] for p in packs]),
            torch.cat([p["x1"] for p in packs]),
        ],
        dim=1,
    )
    n_atoms = torch.cat([p["n_atoms"] for p in packs])
    context = torch.cat([p["context"] for p in packs])
    return waypoints, n_atoms, context


def segment_loss(student, waypoints, node_mask, edge_mask, batch_context):
    """Linear CFM on a random hop. t_clock=(k+u)/3; v* = end-start = dz/du."""
    B = waypoints.shape[0]
    k = torch.randint(0, N_SEG, (B,), device=waypoints.device)
    u = torch.rand(B, 1, device=waypoints.device)
    t = (k.float()[:, None] + u) / N_SEG
    idx = torch.arange(B, device=waypoints.device)
    start, end = waypoints[idx, k], waypoints[idx, k + 1]
    u_b = u[:, None, :]
    xt = (1.0 - u_b) * start + u_b * end
    xt = torch.cat(
        [remove_mean_with_mask(xt[..., :3], node_mask), xt[..., 3:]], -1
    ) * node_mask
    target = (end - start) * node_mask
    pred = student.velocity(xt, t, node_mask, edge_mask, batch_context)
    err = (pred - target) ** 2 * node_mask
    n = node_mask.sum().clamp_min(1)
    loss_x = err[..., :3].sum() / (n * student.n_dims)
    loss_h = err[..., 3:].sum() / (n * student.in_node_nf)
    return loss_x + loss_h


def save_ckpt(path, epoch, avg, best, student, student_ema, opt, gradnorm_queue):
    torch.save(
        {
            "epoch": epoch,
            "avg_loss": avg,
            "best_loss": best,
            "student": student.state_dict(),
            "student_ema": student_ema.state_dict() if student_ema is not None else None,
            "opt": opt.state_dict(),
            "hidden_nf": HIDDEN_NF,
            "lr": LR,
            "batch": BATCH,
            "ema_decay": EMA_DECAY,
            "n_seg": N_SEG,
            "gradnorm_queue": list(gradnorm_queue.items),
        },
        path,
    )


edm_ckpt = torch.load(EDM_WEIGHTS, map_location="cpu", weights_only=False)
norms = {
    k: torch.tensor(v, device=device, dtype=torch.float32)
    for k, v in edm_ckpt.get("context_norms", CONTEXT_NORMS).items()
}

student = EquivariantFlowMatching(
    dynamics=EGNNDynamics(
        in_node_nf=9, context_node_nf=3, hidden_nf=HIDDEN_NF, device=device
    ),
    in_node_nf=8,
).to(device)
student_ema = copy.deepcopy(student).to(device)
for p in student_ema.parameters():
    p.requires_grad_(False)
ema = EMA(EMA_DECAY)
opt = AdamW(student.parameters(), lr=LR, amsgrad=True, weight_decay=1e-12)
gradnorm_queue = Queue()
gradnorm_queue.add(3000.0)

log(
    f"start 3seg device={device} batch={BATCH} hidden_nf={HIDDEN_NF} lr={LR} "
    f"ema={EMA_DECAY} epochs={EPOCHS} adaptive_clip=True"
)

best_loss = float("inf")
epochs_no_improve = 0

for epoch in range(EPOCHS):
    part_i = PART_CYCLE[epoch % len(PART_CYCLE)]
    waypoints, n_atoms, context = load_part_pairs(part_i)
    perm = torch.randperm(waypoints.shape[0])
    waypoints, n_atoms, context = waypoints[perm], n_atoms[perm], context[perm]
    log(f"epoch={epoch} part={part_i + 1} n={waypoints.shape[0]}")

    student.train()
    running, n_steps = 0.0, 0
    pbar = tqdm(
        range(0, waypoints.shape[0] - BATCH + 1, BATCH),
        desc=f"ep{epoch} part{part_i + 1}",
    )
    for i in pbar:
        sl = slice(i, i + BATCH)
        na = n_atoms[sl].to(device=device, dtype=torch.long)
        node_mask, edge_mask = prepare_masks(na, PAD_TO, device)
        ctx = context[sl].to(device=device, dtype=torch.float32)
        normed = (ctx - norms["mean"]) / norms["mad"]
        batch_context = normed.unsqueeze(1).expand(-1, PAD_TO, -1) * node_mask
        wp = waypoints[sl].to(device=device, dtype=torch.float32)

        loss = segment_loss(student, wp, node_mask, edge_mask, batch_context)
        if not torch.isfinite(loss):
            log(f"WARN non-finite epoch={epoch} step={n_steps}")
            continue

        opt.zero_grad(set_to_none=True)
        loss.backward()
        gradient_clipping(student, gradnorm_queue)
        opt.step()
        ema.update_model_average(student_ema, student)

        running += loss.item()
        n_steps += 1
        if n_steps % 20 == 0:
            pbar.set_postfix(loss=f"{running / n_steps:.4f}")

    avg = running / max(n_steps, 1)
    if avg < best_loss:
        best_loss = avg
        save_ckpt(
            CKPT_DIR / f"best_{HIDDEN_NF}_3seg.pt",
            epoch, avg, best_loss, student, student_ema, opt, gradnorm_queue,
        )
        log(f"ckpt best epoch={epoch} avg_loss={avg:.6f}")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    log(
        f"epoch={epoch} part={part_i + 1} steps={n_steps} "
        f"avg_loss={avg:.6f} best_loss={best_loss:.6f} "
        f"grad_q_mean={gradnorm_queue.mean():.1f}"
    )

    if (epoch + 1) % CKPT_EVERY == 0:
        save_ckpt(
            CKPT_DIR / f"latest_{HIDDEN_NF}_3seg.pt",
            epoch, avg, best_loss, student, student_ema, opt, gradnorm_queue,
        )
        log(f"ckpt latest epoch={epoch} avg_loss={avg:.6f}")

    if epochs_no_improve >= EARLY_STOP_PATIENCE:
        log(f"early stop at epoch={epoch} best_loss={best_loss:.6f}")
        break

log(f"done best_loss={best_loss:.6f}")

In [ ]:
"""
Progressive jump distillation: 1 student DDIM step matches 2 teacher DDIM steps.
Faster trajectory = coarser jumps on the teacher manifold (not a Euclidean chord).
After stages, sample with student_ddim(..., n_steps=8).
"""
import copy
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
from torch.optim import AdamW
from tqdm import tqdm

try:
    from src.mlconfgen.egnn import EGNNDynamics
    from src.mlconfgen.equivariant_diffusion import EquivariantDiffusion, PredefinedNoiseSchedule
    from src.mlconfgen.equivariant_flow_matching import EquivariantFlowMatching
    from src.mlconfgen.utils import CONTEXT_NORMS, MAX_N_NODES, remove_mean_with_mask
    from src.mlconfgen.utils.mol_utils import prepare_masks
except ImportError:
    from ml_conformer_generator.src.mlconfgen.egnn import EGNNDynamics
    from ml_conformer_generator.src.mlconfgen.equivariant_diffusion import (
        EquivariantDiffusion, PredefinedNoiseSchedule,
    )
    from ml_conformer_generator.src.mlconfgen.equivariant_flow_matching import EquivariantFlowMatching
    from ml_conformer_generator.src.mlconfgen.utils import CONTEXT_NORMS, MAX_N_NODES, remove_mean_with_mask
    from ml_conformer_generator.src.mlconfgen.utils.mol_utils import prepare_masks

# --------------------- config ---------------------
device = "cuda"
BATCH = 128
HIDDEN_NF = 256              # light; 128 if you must
WARM_START = False           # True only if HIDDEN_NF == 420
TEACHER_T = 100
NOISE_PRECISION = 1e-5
LR = 1e-4
EMA_DECAY = 0.999
PAD_TO = MAX_N_NODES
LAMBDA_X, LAMBDA_H = 1.0, 1.0
PART_CYCLE = [0, 1]
CKPT_EVERY = 2

# each stage: 1 student step spans jump_k / T, matching 2 DDIM evals of current teacher
STAGES = [
    {"name": "k2",  "epochs": 6, "jump_k": 2},
    {"name": "k4",  "epochs": 6, "jump_k": 4},
    {"name": "k8",  "epochs": 6, "jump_k": 8},
    {"name": "k12", "epochs": 8, "jump_k": 12},   # ~8 NFE over [0, 1]
]

PAIR_DIR = Path("./teacher_pairs/teacher_pairs")
EDM_WEIGHTS = "edm_moi_chembl_15_39.pt"
CKPT_DIR = Path("./checkpoints_fm")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = CKPT_DIR / f"train_{HIDDEN_NF}_jump.log"


class EMA:
    def __init__(self, beta: float):
        self.beta = beta

    def update_model_average(self, ma_model, current_model):
        for cur, ma in zip(current_model.parameters(), ma_model.parameters()):
            ma.data = self.update_average(ma.data, cur.data)

    def update_average(self, old, new):
        if old is None:
            return new
        return old * self.beta + (1.0 - self.beta) * new


class Queue:
    def __init__(self, max_len: int = 50):
        self.items = []
        self.max_len = max_len

    def __len__(self):
        return len(self.items)

    def add(self, item):
        self.items.insert(0, item)
        if len(self) > self.max_len:
            self.items.pop()

    def mean(self):
        return float(np.mean(self.items))

    def std(self):
        return float(np.std(self.items))


def gradient_clipping(model, gradnorm_queue: Queue):
    max_grad_norm = 1.5 * gradnorm_queue.mean() + 2.0 * gradnorm_queue.std()
    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(), max_norm=max_grad_norm, norm_type=2.0
    )
    gn = float(grad_norm)
    if gn > max_grad_norm:
        gradnorm_queue.add(float(max_grad_norm))
    else:
        gradnorm_queue.add(gn)


def log(msg: str) -> None:
    line = f"{datetime.now().isoformat(timespec='seconds')}  {msg}"
    print(line)
    with open(LOG_PATH, "a") as f:
        f.write(line + "\n")


def com_project(z, node_mask):
    return torch.cat(
        [remove_mean_with_mask(z[..., :3], node_mask), z[..., 3:]], dim=-1
    ) * node_mask


def load_part_pairs(part_i: int):
    paths = sorted(PAIR_DIR.glob(f"part_{part_i + 1}_shard*.pt"))
    if not paths:
        paths = sorted(PAIR_DIR.glob(f"part{part_i + 1}_shard*.pt"))
    if not paths:
        paths = sorted(PAIR_DIR.glob("shard*.pt"))
        if part_i != 0:
            raise FileNotFoundError(f"no part shards in {PAIR_DIR}")
    if not paths:
        raise FileNotFoundError(PAIR_DIR)
    packs = [torch.load(p, map_location="cpu", weights_only=False) for p in paths]
    return (
        torch.cat([p["z_T"] for p in packs]),
        torch.cat([p["x1"] for p in packs]),
        torch.cat([p["n_atoms"] for p in packs]),
        torch.cat([p["context"] for p in packs]),
    )


def gamma_and_dgamma(t, sched):
    T = sched.timesteps
    u = t * T
    i0 = u.floor().long().clamp(0, T - 1)
    i1 = i0 + 1
    frac = (u - i0.float()).clamp(0.0, 1.0)
    g = sched.gamma
    g0, g1 = g[i0], g[i1]
    return g0 + frac * (g1 - g0), (g1 - g0) * T


def alpha_sigma(t, sched, like):
    gamma, _ = gamma_and_dgamma(t, sched)
    alpha = torch.sqrt(torch.sigmoid(-gamma))
    sigma = torch.sqrt(torch.sigmoid(gamma))
    view = (like.shape[0], 1, 1)
    return alpha.view(view), sigma.view(view)


def ddim_step(eps_fn, z, t, s, sched, node_mask, edge_mask, context):
    """Deterministic DDIM: z_t, ε_θ → z_s. t,s are (B,1) EDM times."""
    eps = eps_fn(z, t, node_mask, edge_mask, context)
    a_t, s_t = alpha_sigma(t, sched, z)
    a_s, s_s = alpha_sigma(s, sched, z)
    x_hat = com_project((z - s_t * eps) / a_t.clamp_min(1e-4), node_mask)
    return com_project(a_s * x_hat + s_s * eps, node_mask)


def teacher_two_jump(eps_fn, z, t, s, sched, node_mask, edge_mask, context):
    t_mid = 0.5 * (t + s)
    z_mid = ddim_step(eps_fn, z, t, t_mid, sched, node_mask, edge_mask, context)
    return ddim_step(eps_fn, z_mid, t_mid, s, sched, node_mask, edge_mask, context)


def split_mse(pred, target, node_mask, n_dims=3, n_h=8):
    err = (pred - target) ** 2 * node_mask
    n = node_mask.sum().clamp_min(1)
    loss_x = err[..., :3].sum() / (n * n_dims)
    loss_h = err[..., 3:].sum() / (n * n_h)
    return LAMBDA_X * loss_x + LAMBDA_H * loss_h, loss_x.detach(), loss_h.detach()


def save_ckpt(path, epoch, avg, best, stage, jump_k, student, student_ema, opt, gradnorm_queue):
    torch.save(
        {
            "epoch": epoch,
            "avg_loss": avg,
            "best_loss": best,
            "stage": stage,
            "jump_k": jump_k,
            "student": student.state_dict(),
            "student_ema": student_ema.state_dict(),
            "opt": opt.state_dict(),
            "hidden_nf": HIDDEN_NF,
            "teacher_T": TEACHER_T,
            "gradnorm_queue": list(gradnorm_queue.items),
        },
        path,
    )


@torch.inference_mode()
def student_ddim(student, node_mask, edge_mask, context, sched, n_steps=8):
    """Few-NFE sample. t: 1-1/T → 0."""
    student.eval()
    B, N = node_mask.shape[:2]
    z = student.sample_combined_position_feature_noise(B, N, node_mask)
    ts = torch.linspace(1.0 - 1.0 / TEACHER_T, 0.0, n_steps + 1, device=z.device)
    def eps_fn(z_, t_, nm, em, ctx):
        return student.velocity(z_, t_, nm, em, ctx)
    for i in range(n_steps):
        t0 = ts[i].expand(B, 1)
        t1 = ts[i + 1].expand(B, 1)
        z = ddim_step(eps_fn, z, t0, t1, sched, node_mask, edge_mask, context)
    x = z[..., :3]
    h = torch.nn.functional.one_hot(z[..., 3:].argmax(-1), num_classes=8).float() * node_mask
    return remove_mean_with_mask(x, node_mask), h


# --------------------- models ---------------------
edm_ckpt = torch.load(EDM_WEIGHTS, map_location="cpu", weights_only=False)
norms = {
    k: torch.tensor(v, device=device, dtype=torch.float32)
    for k, v in edm_ckpt.get("context_norms", CONTEXT_NORMS).items()
}

sched = PredefinedNoiseSchedule(timesteps=TEACHER_T, precision=NOISE_PRECISION).to(device)

teacher_dyn = EGNNDynamics(in_node_nf=9, context_node_nf=3, hidden_nf=420, device=device)
teacher = EquivariantDiffusion(
    dynamics=teacher_dyn, in_node_nf=8, timesteps=1000, noise_precision=NOISE_PRECISION
)
teacher.load_state_dict(edm_ckpt["state_dict"])
teacher.gamma = sched
teacher.T = TEACHER_T
teacher.to(device).eval()
for p in teacher.parameters():
    p.requires_grad_(False)

student_dyn = EGNNDynamics(
    in_node_nf=9, context_node_nf=3, hidden_nf=HIDDEN_NF, device=device
)
if WARM_START:
    if HIDDEN_NF != 420:
        raise ValueError("WARM_START requires HIDDEN_NF=420")
    student_dyn.load_state_dict(teacher.dynamics.state_dict())
student = EquivariantFlowMatching(dynamics=student_dyn, in_node_nf=8).to(device)
student_ema = copy.deepcopy(student).to(device)
for p in student_ema.parameters():
    p.requires_grad_(False)
ema = EMA(EMA_DECAY)
opt = AdamW(student.parameters(), lr=LR, amsgrad=True, weight_decay=1e-12)
gradnorm_queue = Queue()
gradnorm_queue.add(3000.0)

def edm_eps(z, t, nm, em, ctx):
    return teacher.phi(z, t, nm, em, ctx)

def fm_eps(model):
    def _f(z, t, nm, em, ctx):
        return model.velocity(z, t, nm, em, ctx)
    return _f

jump_teacher_eps = edm_eps  # replaced by student_ema after each stage

log(
    f"start jump-distill device={device} hidden_nf={HIDDEN_NF} warm={WARM_START} "
    f"batch={BATCH} stages={[s['jump_k'] for s in STAGES]}"
)

global_epoch = 0
for stage in STAGES:
    jump_k = stage["jump_k"]
    jump_dt = jump_k / TEACHER_T
    best_loss = float("inf")
    log(f"=== stage={stage['name']} jump_k={jump_k} dt={jump_dt:.4f} ===")

    for local_ep in range(stage["epochs"]):
        part_i = PART_CYCLE[global_epoch % len(PART_CYCLE)]
        z_T, x1, n_atoms, context = load_part_pairs(part_i)
        perm = torch.randperm(z_T.shape[0])
        z_T, x1, n_atoms, context = z_T[perm], x1[perm], n_atoms[perm], context[perm]
        log(f"epoch={global_epoch} stage={stage['name']} part={part_i + 1} n={z_T.shape[0]}")

        student.train()
        running, run_x, run_h, n_steps = 0.0, 0.0, 0.0, 0
        pbar = tqdm(
            range(0, z_T.shape[0] - BATCH + 1, BATCH),
            desc=f"{stage['name']} ep{global_epoch}",
        )
        for i in pbar:
            sl = slice(i, i + BATCH)
            na = n_atoms[sl].to(device=device, dtype=torch.long)
            node_mask, edge_mask = prepare_masks(na, PAD_TO, device)
            ctx = context[sl].to(device=device, dtype=torch.float32)
            batch_context = (
                (ctx - norms["mean"]) / norms["mad"]
            ).unsqueeze(1).expand(-1, PAD_TO, -1) * node_mask
            x0_b = z_T[sl].to(device=device, dtype=torch.float32)
            x1_b = x1[sl].to(device=device, dtype=torch.float32)

            B = x1_b.shape[0]
            # t ∈ [jump_dt + 1/T, 1 - 1/T] so s ≥ 1/T
            t = torch.rand(B, 1, device=device) * (1.0 - jump_dt - 2.0 / TEACHER_T) + (
                jump_dt + 1.0 / TEACHER_T
            )
            s = t - jump_dt
            a_t, sig_t = alpha_sigma(t, sched, x1_b)
            zt = com_project(a_t * x1_b + sig_t * x0_b, node_mask)

            with torch.no_grad():
                eps_tgt = jump_teacher_eps(
                    zt, t, node_mask, edge_mask, batch_context
                )
                z_tgt = teacher_two_jump(
                    jump_teacher_eps, zt, t, s, sched, node_mask, edge_mask, batch_context
                )
            eps_pred = student.velocity(zt, t, node_mask, edge_mask, batch_context)
            z_pred = ddim_step(
                fm_eps(student), zt, t, s, sched, node_mask, edge_mask, batch_context
            )
            # k2 hops leave z_s ≈ z_t, so z-MSE is vacuous. Supervise ε (what DDIM uses).
            loss_e, loss_x, loss_h = split_mse(eps_pred, eps_tgt, node_mask)
            loss_z, _, _ = split_mse(z_pred, z_tgt, node_mask)
            loss = loss_e + (jump_k / 2.0) * loss_z
            if not torch.isfinite(loss):
                log(f"WARN non-finite epoch={global_epoch} step={n_steps}")
                continue

            opt.zero_grad(set_to_none=True)
            loss.backward()
            gradient_clipping(student, gradnorm_queue)
            opt.step()
            ema.update_model_average(student_ema, student)

            running += loss.item()
            run_x += loss_x.item()
            run_h += loss_h.item()
            n_steps += 1
            if n_steps % 20 == 0:
                pbar.set_postfix(
                    loss=f"{running / n_steps:.4f}",
                    lx=f"{run_x / n_steps:.4f}",
                    lh=f"{run_h / n_steps:.4f}",
                )

        avg = running / max(n_steps, 1)
        avg_x = run_x / max(n_steps, 1)
        avg_h = run_h / max(n_steps, 1)
        if avg < best_loss:
            best_loss = avg
            save_ckpt(
                CKPT_DIR / f"best_{HIDDEN_NF}_jump_{stage['name']}.pt",
                global_epoch, avg, best_loss, stage["name"], jump_k,
                student, student_ema, opt, gradnorm_queue,
            )
            log(f"ckpt best stage={stage['name']} epoch={global_epoch} avg_loss={avg:.6f}")

        log(
            f"epoch={global_epoch} stage={stage['name']} jump_k={jump_k} "
            f"avg_loss={avg:.6f} loss_x={avg_x:.6f} loss_h={avg_h:.6f} "
            f"best={best_loss:.6f}"
        )
        if (local_ep + 1) % CKPT_EVERY == 0:
            save_ckpt(
                CKPT_DIR / f"latest_{HIDDEN_NF}_jump.pt",
                global_epoch, avg, best_loss, stage["name"], jump_k,
                student, student_ema, opt, gradnorm_queue,
            )
        global_epoch += 1

    # next stage: 2 jumps of this student replace 2 jumps of EDM
    jump_teacher = copy.deepcopy(student_ema).to(device)
    jump_teacher.eval()
    for p in jump_teacher.parameters():
        p.requires_grad_(False)
    jump_teacher_eps = fm_eps(jump_teacher)
    log(f"promoted student_ema → jump teacher for next stage")

log("done jump-distill")
# sample: x, h = student_ddim(student_ema, node_mask, edge_mask, context, sched, n_steps=8)


In [ ]:
"""
Coarse interpolant FM: predict the next noised molecule z_s, not ε, not DDIM.
Sampler is Euler z ← z + v. No 1/α. NFE must match training.
"""
import copy
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
from torch.optim import AdamW
from tqdm import tqdm

try:
    from src.mlconfgen.egnn import EGNNDynamics
    from src.mlconfgen.equivariant_diffusion import PredefinedNoiseSchedule
    from src.mlconfgen.equivariant_flow_matching import EquivariantFlowMatching
    from src.mlconfgen.utils import CONTEXT_NORMS, MAX_N_NODES, remove_mean_with_mask
    from src.mlconfgen.utils.mol_utils import prepare_masks
except ImportError:
    from ml_conformer_generator.src.mlconfgen.egnn import EGNNDynamics
    from ml_conformer_generator.src.mlconfgen.equivariant_diffusion import PredefinedNoiseSchedule
    from ml_conformer_generator.src.mlconfgen.equivariant_flow_matching import EquivariantFlowMatching
    from ml_conformer_generator.src.mlconfgen.utils import CONTEXT_NORMS, MAX_N_NODES, remove_mean_with_mask
    from ml_conformer_generator.src.mlconfgen.utils.mol_utils import prepare_masks

device = "cuda"
BATCH = 256
HIDDEN_NF = 128
NFE = 8                 # train and sample with this many Euler steps
TEACHER_T = 100
LR = 1e-4
EPOCHS = 20
EMA_DECAY = 0.999
PAD_TO = MAX_N_NODES
PART_CYCLE = [0, 1]
EARLY_STOP_PATIENCE = 6
MIN_DELTA = 0.005

PAIR_DIR = Path("./teacher_pairs/teacher_pairs")
EDM_WEIGHTS = "edm_moi_chembl_15_39.pt"
CKPT_DIR = Path("./checkpoints_fm")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = CKPT_DIR / f"train_{HIDDEN_NF}_grid{NFE}.log"
DT = 1.0 / NFE


class EMA:
    def __init__(self, beta):
        self.beta = beta

    def update_model_average(self, ma, cur):
        for c, m in zip(cur.parameters(), ma.parameters()):
            m.data = m.data * self.beta + (1 - self.beta) * c.data


class Queue:
    def __init__(self, max_len=50):
        self.items, self.max_len = [], max_len

    def __len__(self):
        return len(self.items)

    def add(self, x):
        self.items.insert(0, x)
        if len(self) > self.max_len:
            self.items.pop()

    def mean(self):
        return float(np.mean(self.items))

    def std(self):
        return float(np.std(self.items))


def gradient_clipping(model, q):
    mx = 1.5 * q.mean() + 2.0 * q.std()
    gn = float(torch.nn.utils.clip_grad_norm_(model.parameters(), mx))
    q.add(mx if gn > mx else gn)


def log(msg):
    line = f"{datetime.now().isoformat(timespec='seconds')}  {msg}"
    print(line)
    with open(LOG_PATH, "a") as f:
        f.write(line + "\n")


def com_project(z, node_mask):
    return torch.cat(
        [remove_mean_with_mask(z[..., :3], node_mask), z[..., 3:]], -1
    ) * node_mask


def load_part_pairs(part_i):
    paths = sorted(PAIR_DIR.glob(f"part_{part_i + 1}_shard*.pt")) or sorted(
        PAIR_DIR.glob(f"part{part_i + 1}_shard*.pt")
    )
    if not paths:
        paths = sorted(PAIR_DIR.glob("shard*.pt"))
        if part_i != 0:
            raise FileNotFoundError(PAIR_DIR)
    packs = [torch.load(p, map_location="cpu", weights_only=False) for p in paths]
    return (
        torch.cat([p["z_T"] for p in packs]),
        torch.cat([p["x1"] for p in packs]),
        torch.cat([p["n_atoms"] for p in packs]),
        torch.cat([p["context"] for p in packs]),
    )


def alpha_sigma(t, sched, like):
    g = sched.gamma[(t * sched.timesteps).round().long().clamp(0, sched.timesteps)]
    a = torch.sqrt(torch.sigmoid(-g)).view(like.shape[0], 1, 1)
    s = torch.sqrt(torch.sigmoid(g)).view(like.shape[0], 1, 1)
    return a, s


def interpolant(x1, zT, t, sched, node_mask):
    a, s = alpha_sigma(t, sched, x1)
    return com_project(a * x1 + s * zT, node_mask)


edm_ckpt = torch.load(EDM_WEIGHTS, map_location="cpu", weights_only=False)
norms = {
    k: torch.tensor(v, device=device, dtype=torch.float32)
    for k, v in edm_ckpt.get("context_norms", CONTEXT_NORMS).items()
}
sched = PredefinedNoiseSchedule(timesteps=TEACHER_T, precision=1e-5).to(device)

student = EquivariantFlowMatching(
    dynamics=EGNNDynamics(
        in_node_nf=9, context_node_nf=3, hidden_nf=HIDDEN_NF, device=device
    ),
    in_node_nf=8,
).to(device)
student_ema = copy.deepcopy(student).to(device)
for p in student_ema.parameters():
    p.requires_grad_(False)
ema = EMA(EMA_DECAY)
opt = AdamW(student.parameters(), lr=LR, amsgrad=True, weight_decay=1e-12)
q = Queue()
q.add(3000.0)

log(f"start grid-FM NFE={NFE} dt={DT} hidden={HIDDEN_NF} batch={BATCH}")
best, no_imp = float("inf"), 0

for epoch in range(EPOCHS):
    part_i = PART_CYCLE[epoch % len(PART_CYCLE)]
    z_T, x1, n_atoms, context = load_part_pairs(part_i)
    perm = torch.randperm(z_T.shape[0])
    z_T, x1, n_atoms, context = z_T[perm], x1[perm], n_atoms[perm], context[perm]
    log(f"epoch={epoch} part={part_i + 1} n={z_T.shape[0]}")
    student.train()
    run, rx, rh, ns = 0.0, 0.0, 0.0, 0
    pbar = tqdm(range(0, z_T.shape[0] - BATCH + 1, BATCH), desc=f"ep{epoch}")
    for i in pbar:
        sl = slice(i, i + BATCH)
        na = n_atoms[sl].to(device=device, dtype=torch.long)
        node_mask, edge_mask = prepare_masks(na, PAD_TO, device)
        ctx = context[sl].to(device=device, dtype=torch.float32)
        bctx = ((ctx - norms["mean"]) / norms["mad"]).unsqueeze(1).expand(-1, PAD_TO, -1) * node_mask
        zT_b = z_T[sl].to(device=device, dtype=torch.float32)
        x1_b = x1[sl].to(device=device, dtype=torch.float32)
        B = x1_b.shape[0]

        t = torch.rand(B, 1, device=device) * (1.0 - DT - 1.0 / TEACHER_T) + DT
        s = t - DT
        zt = interpolant(x1_b, zT_b, t, sched, node_mask)
        zs = interpolant(x1_b, zT_b, s, sched, node_mask)
        pred = com_project(
            zt + student.velocity(zt, t, node_mask, edge_mask, bctx), node_mask
        )
        err = (pred - zs) ** 2 * node_mask
        n = node_mask.sum().clamp_min(1)
        loss_x = err[..., :3].sum() / (n * 3)
        loss_h = err[..., 3:].sum() / (n * 8)
        loss = loss_x + loss_h
        if not torch.isfinite(loss):
            continue
        opt.zero_grad(set_to_none=True)
        loss.backward()
        gradient_clipping(student, q)
        opt.step()
        ema.update_model_average(student_ema, student)
        run += loss.item()
        rx += loss_x.item()
        rh += loss_h.item()
        ns += 1
        if ns % 20 == 0:
            pbar.set_postfix(loss=f"{run / ns:.4f}", lx=f"{rx / ns:.4f}", lh=f"{rh / ns:.4f}")

    avg = run / max(ns, 1)
    log(f"epoch={epoch} avg={avg:.6f} lx={rx / max(ns, 1):.6f} lh={rh / max(ns, 1):.6f}")
    ckpt = {
        "epoch": epoch,
        "avg_loss": avg,
        "nfe": NFE,
        "student": student.state_dict(),
        "student_ema": student_ema.state_dict(),
        "hidden_nf": HIDDEN_NF,
        "opt": opt.state_dict(),
    }
    torch.save(ckpt, CKPT_DIR / f"latest_{HIDDEN_NF}_grid{NFE}.pt")
    if avg < best - MIN_DELTA:
        best, no_imp = avg, 0
        torch.save(ckpt, CKPT_DIR / f"best_{HIDDEN_NF}_grid{NFE}.pt")
        log(f"ckpt best avg={avg:.6f}")
    else:
        no_imp += 1
        if avg < best:
            best = avg
    if no_imp >= EARLY_STOP_PATIENCE:
        log(f"early stop epoch={epoch}")
        break
log(f"done best={best:.6f}")


In [ ]:
"""
Progressive distillation of the EDM into a v-parametrized FM student.

Stage A: v-distill the teacher posterior. Fresh ε (NO paired z_T as noise),
         target from teacher.phi. Gate: 50-step student DDIM ≈ teacher.
Stage B: halve steps 32 → 16 → 8. One student DDIM step (Δ=1/N) matches two
         half-steps of the current jump teacher. Target is the IMPLIED clean
         latent x̃ (Salimans–Ho inversion), so small hops are not vacuous.

v-parametrization: v* = αε − σx;  x̂ = αz − σv;  ε̂ = σz + αv.
No division by α or σ anywhere in the student path.
"""
import copy
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
from torch.optim import AdamW
from tqdm import tqdm

try:
    from src.mlconfgen.egnn import EGNNDynamics
    from src.mlconfgen.equivariant_diffusion import EquivariantDiffusion, PredefinedNoiseSchedule
    from src.mlconfgen.equivariant_flow_matching import EquivariantFlowMatching
    from src.mlconfgen.utils import CONTEXT_NORMS, MAX_N_NODES, remove_mean_with_mask
    from src.mlconfgen.utils.mol_utils import prepare_masks
except ImportError:
    from ml_conformer_generator.src.mlconfgen.egnn import EGNNDynamics
    from ml_conformer_generator.src.mlconfgen.equivariant_diffusion import (
        EquivariantDiffusion, PredefinedNoiseSchedule,
    )
    from ml_conformer_generator.src.mlconfgen.equivariant_flow_matching import EquivariantFlowMatching
    from ml_conformer_generator.src.mlconfgen.utils import CONTEXT_NORMS, MAX_N_NODES, remove_mean_with_mask
    from ml_conformer_generator.src.mlconfgen.utils.mol_utils import prepare_masks

# --------------------- config ---------------------
device = "cuda"
BATCH = 128
HIDDEN_NF = 420          # Stage A/B on 420. Shrink width only AFTER 8-NFE works.
WARM_START = True        # copy teacher.dynamics (requires 420)
TEACHER_T = 100
NOISE_PRECISION = 1e-5
LR = 1e-4
EMA_DECAY = 0.999
PAD_TO = MAX_N_NODES
PART_CYCLE = [0, 1]

STAGE_A_EPOCHS = 4
PD_STAGES = [(32, 4), (16, 4), (8, 6)]   # (grid N, epochs)

PAIR_DIR = Path("./teacher_pairs/teacher_pairs")
EDM_WEIGHTS = "edm_moi_chembl_15_39.pt"
CKPT_DIR = Path("./checkpoints_fm")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = CKPT_DIR / f"train_{HIDDEN_NF}_pd.log"


# --------------------- boilerplate ---------------------
class EMA:
    def __init__(self, beta):
        self.beta = beta

    def update_model_average(self, ma, cur):
        for c, m in zip(cur.parameters(), ma.parameters()):
            m.data = m.data * self.beta + (1 - self.beta) * c.data


class Queue:
    def __init__(self, max_len=50):
        self.items, self.max_len = [], max_len

    def __len__(self):
        return len(self.items)

    def add(self, x):
        self.items.insert(0, x)
        if len(self) > self.max_len:
            self.items.pop()

    def mean(self):
        return float(np.mean(self.items))

    def std(self):
        return float(np.std(self.items))


def gradient_clipping(model, q):
    mx = 1.5 * q.mean() + 2.0 * q.std()
    gn = float(torch.nn.utils.clip_grad_norm_(model.parameters(), mx))
    q.add(mx if gn > mx else gn)


def log(msg):
    line = f"{datetime.now().isoformat(timespec='seconds')}  {msg}"
    print(line)
    with open(LOG_PATH, "a") as f:
        f.write(line + "\n")


def com_project(z, node_mask):
    return torch.cat(
        [remove_mean_with_mask(z[..., :3], node_mask), z[..., 3:]], -1
    ) * node_mask


def load_part_pairs(part_i):
    paths = sorted(PAIR_DIR.glob(f"part_{part_i + 1}_shard*.pt")) or sorted(
        PAIR_DIR.glob(f"part{part_i + 1}_shard*.pt")
    )
    if not paths:
        paths = sorted(PAIR_DIR.glob("shard*.pt"))
        if part_i != 0:
            raise FileNotFoundError(PAIR_DIR)
    packs = [torch.load(p, map_location="cpu", weights_only=False) for p in paths]
    return (
        torch.cat([p["x1"] for p in packs]),          # z_T is NOT used as noise
        torch.cat([p["n_atoms"] for p in packs]),
        torch.cat([p["context"] for p in packs]),
    )


# --------------------- schedule (continuous t via γ interp) ---------------------
def alpha_sigma(t, sched, like):
    T = sched.timesteps
    u = (t * T).clamp(0, T)
    i0 = u.floor().long().clamp(0, T - 1)
    frac = (u - i0.float()).clamp(0.0, 1.0)
    g = sched.gamma
    gamma = g[i0] + frac * (g[i0 + 1] - g[i0])
    a = torch.sqrt(torch.sigmoid(-gamma)).view(like.shape[0], 1, 1)
    s = torch.sqrt(torch.sigmoid(gamma)).view(like.shape[0], 1, 1)
    return a, s


# --------------------- v-parametrization ---------------------
def x_eps_from_v(z, v, a, s):
    x = a * z - s * v
    eps = s * z + a * v
    return x, eps


def v_from_eps(z, eps, a, s):
    """Teacher-side only: ε is accurate, so 1/α (clamped) is safe here."""
    x = (z - s * eps) / a.clamp_min(1e-3)
    return a * eps - s * x, x


def ddim_v(z, v, a_t, s_t, a_s, s_s, node_mask):
    """z_t → z_s using v. Multiplications only."""
    x, eps = x_eps_from_v(z, v, a_t, s_t)
    return com_project(a_s * x + s_s * eps, node_mask)


def pd_x_target(z_t, z_s_tilde, a_t, s_t, a_s, s_s):
    """Salimans–Ho: invert one DDIM step to the implied clean latent.
    x̃ = (z̃_s − (σ_s/σ_t) z_t) / (α_s − α_t σ_s/σ_t).
    Small denominators RESCALE the target instead of letting the loss vanish."""
    ratio = s_s / s_t.clamp_min(1e-4)
    denom = (a_s - a_t * ratio).clamp_min(1e-4)
    return (z_s_tilde - ratio * z_t) / denom


def snr_weight(a, s):
    """Truncated SNR: w = max(α²/σ², 1). ε-loss at low t, x-loss at high t."""
    return torch.clamp(a * a / (s * s).clamp_min(1e-8), min=1.0)


def x_loss(x_pred, x_tgt, w, node_mask):
    err = (x_pred - x_tgt) ** 2 * node_mask * w
    n = node_mask.sum().clamp_min(1)
    lx = err[..., :3].sum() / (n * 3)
    lh = err[..., 3:].sum() / (n * 8)
    return lx + lh, lx.detach(), lh.detach()


# --------------------- models ---------------------
edm_ckpt = torch.load(EDM_WEIGHTS, map_location="cpu", weights_only=False)
norms = {
    k: torch.tensor(v, device=device, dtype=torch.float32)
    for k, v in edm_ckpt.get("context_norms", CONTEXT_NORMS).items()
}
sched = PredefinedNoiseSchedule(timesteps=TEACHER_T, precision=NOISE_PRECISION).to(device)

teacher_dyn = EGNNDynamics(in_node_nf=9, context_node_nf=3, hidden_nf=420, device=device)
teacher = EquivariantDiffusion(
    dynamics=teacher_dyn, in_node_nf=8, timesteps=1000, noise_precision=NOISE_PRECISION
)
teacher.load_state_dict(edm_ckpt["state_dict"])
teacher.gamma = sched
teacher.T = TEACHER_T
teacher.to(device).eval()
for p in teacher.parameters():
    p.requires_grad_(False)

student_dyn = EGNNDynamics(
    in_node_nf=9, context_node_nf=3, hidden_nf=HIDDEN_NF, device=device
)
if WARM_START:
    assert HIDDEN_NF == 420, "WARM_START requires HIDDEN_NF=420"
    student_dyn.load_state_dict(teacher.dynamics.state_dict())
student = EquivariantFlowMatching(dynamics=student_dyn, in_node_nf=8).to(device)
student_ema = copy.deepcopy(student).to(device)
for p in student_ema.parameters():
    p.requires_grad_(False)
ema = EMA(EMA_DECAY)
opt = AdamW(student.parameters(), lr=LR, amsgrad=True, weight_decay=1e-12)
q = Queue()
q.add(3000.0)


def make_batch(x1_all, n_atoms_all, ctx_all, sl):
    na = n_atoms_all[sl].to(device=device, dtype=torch.long)
    nm, em = prepare_masks(na, PAD_TO, device)
    ctx = ctx_all[sl].to(device=device, dtype=torch.float32)
    bctx = ((ctx - norms["mean"]) / norms["mad"]).unsqueeze(1).expand(-1, PAD_TO, -1) * nm
    x1_b = com_project(x1_all[sl].to(device=device, dtype=torch.float32), nm)
    return x1_b, nm, em, bctx


def run_epochs(tag, epochs, step_fn, ckpt_name):
    global global_epoch
    best = float("inf")
    for _ in range(epochs):
        part_i = PART_CYCLE[global_epoch % len(PART_CYCLE)]
        x1_all, na_all, ctx_all = load_part_pairs(part_i)
        perm = torch.randperm(x1_all.shape[0])
        x1_all, na_all, ctx_all = x1_all[perm], na_all[perm], ctx_all[perm]
        student.train()
        run, rx, rh, ns = 0.0, 0.0, 0.0, 0
        pbar = tqdm(range(0, x1_all.shape[0] - BATCH + 1, BATCH), desc=f"{tag} ep{global_epoch}")
        for i in pbar:
            x1_b, nm, em, bctx = make_batch(x1_all, na_all, ctx_all, slice(i, i + BATCH))
            loss, lx, lh = step_fn(x1_b, nm, em, bctx)
            if not torch.isfinite(loss):
                log(f"WARN non-finite {tag} ep={global_epoch}")
                continue
            opt.zero_grad(set_to_none=True)
            loss.backward()
            gradient_clipping(student, q)
            opt.step()
            ema.update_model_average(student_ema, student)
            run += loss.item(); rx += lx.item(); rh += lh.item(); ns += 1
            if ns % 20 == 0:
                pbar.set_postfix(loss=f"{run/ns:.4f}", lx=f"{rx/ns:.4f}", lh=f"{rh/ns:.4f}")
        avg = run / max(ns, 1)
        log(f"{tag} epoch={global_epoch} avg={avg:.6f} lx={rx/max(ns,1):.6f} lh={rh/max(ns,1):.6f}")
        ckpt = {
            "epoch": global_epoch, "avg_loss": avg, "stage": tag,
            "nfe": current_nfe, "param": "v", "hidden_nf": HIDDEN_NF,
            "student": student.state_dict(), "student_ema": student_ema.state_dict(),
            "opt": opt.state_dict(),
        }
        torch.save(ckpt, CKPT_DIR / f"latest_{HIDDEN_NF}_pd.pt")
        if avg < best:
            best = avg
            torch.save(ckpt, CKPT_DIR / ckpt_name)
            log(f"ckpt best {tag} avg={avg:.6f}")
        global_epoch += 1


# --------------------- Stage A: v-distill teacher posterior ---------------------
def stage_a_step(x1_b, nm, em, bctx):
    B, N = x1_b.shape[:2]
    t = torch.rand(B, 1, device=device) * (1.0 - 2.0 / TEACHER_T) + 1.0 / TEACHER_T
    eps = student.sample_combined_position_feature_noise(B, N, nm)   # FRESH noise
    a_t, s_t = alpha_sigma(t, sched, x1_b)
    zt = com_project(a_t * x1_b + s_t * eps, nm)
    with torch.no_grad():
        eps_tea = teacher.phi(zt, t, nm, em, bctx)
        v_tgt, x_tgt = v_from_eps(zt, eps_tea, a_t, s_t)
    v_pred = student.velocity(zt, t, nm, em, bctx)
    x_pred, _ = x_eps_from_v(zt, v_pred, a_t, s_t)
    return x_loss(x_pred, x_tgt, snr_weight(a_t, s_t), nm)


global_epoch = 0
current_nfe = 50
log(f"=== Stage A: v-distill  hidden={HIDDEN_NF} warm={WARM_START} ===")
run_epochs("A", STAGE_A_EPOCHS, stage_a_step, f"best_{HIDDEN_NF}_vdistill.pt")
log("GATE: sample best_vdistill with 50-step v-DDIM before trusting Stage B.")


# --------------------- Stage B: progressive halving ---------------------
jump_teacher = copy.deepcopy(student_ema).to(device)
jump_teacher.eval()
for p in jump_teacher.parameters():
    p.requires_grad_(False)

for N, n_epochs in PD_STAGES:
    current_nfe = N
    dt = 1.0 / N

    def stage_b_step(x1_b, nm, em, bctx, N=N, dt=dt):
        B, Nn = x1_b.shape[:2]
        i = torch.randint(1, N + 1, (B, 1), device=device).float()
        t = i * dt
        s = t - dt
        tm = t - 0.5 * dt
        eps = student.sample_combined_position_feature_noise(B, Nn, nm)
        a_t, s_t = alpha_sigma(t, sched, x1_b)
        a_m, s_m = alpha_sigma(tm, sched, x1_b)
        a_s, s_s = alpha_sigma(s, sched, x1_b)
        zt = com_project(a_t * x1_b + s_t * eps, nm)
        with torch.no_grad():
            v1 = jump_teacher.velocity(zt, t, nm, em, bctx)
            zm = ddim_v(zt, v1, a_t, s_t, a_m, s_m, nm)
            v2 = jump_teacher.velocity(zm, tm, nm, em, bctx)
            zs = ddim_v(zm, v2, a_m, s_m, a_s, s_s, nm)
            x_tgt = com_project(pd_x_target(zt, zs, a_t, s_t, a_s, s_s), nm)
        v_pred = student.velocity(zt, t, nm, em, bctx)
        x_pred, _ = x_eps_from_v(zt, v_pred, a_t, s_t)
        return x_loss(x_pred, x_tgt, snr_weight(a_t, s_t), nm)

    log(f"=== Stage B: N={N} (Δt={dt:.4f}) ===")
    run_epochs(f"N{N}", n_epochs, stage_b_step, f"best_{HIDDEN_NF}_pd_N{N}.pt")

    jump_teacher = copy.deepcopy(student_ema).to(device)
    jump_teacher.eval()
    for p in jump_teacher.parameters():
        p.requires_grad_(False)
    log(f"promoted student_ema → jump teacher after N={N}")

log("done PD. Sample best_pd_N8 with 8-step v-DDIM (check_sampling).")


In [ ]:
"""
Stage C: one more halving, N=8 -> N=4 (dt=0.25).

Self-contained continuation: reloads everything from best_420_pd_N8.pt.
Two changes vs Stage B, based on the N8 run:
  - jump teacher = NON-EMA student weights (EMA at 0.999 never caught up:
    each stage ran only ~160-480 optimizer steps vs an ~1000-step EMA window).
  - EMA decay 0.99 (~100-step window) so the saved EMA is actually usable.

Gate visually with check_sampling at 4 NFE. Expect avg roughly 2x the N8
level (~0.008-0.012); what matters is whether molecules survive dt=0.25.
"""
import copy
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
from torch.optim import AdamW
from tqdm import tqdm

try:
    from src.mlconfgen.egnn import EGNNDynamics
    from src.mlconfgen.equivariant_diffusion import PredefinedNoiseSchedule
    from src.mlconfgen.equivariant_flow_matching import EquivariantFlowMatching
    from src.mlconfgen.utils import CONTEXT_NORMS, MAX_N_NODES, remove_mean_with_mask
    from src.mlconfgen.utils.mol_utils import prepare_masks
except ImportError:
    from ml_conformer_generator.src.mlconfgen.egnn import EGNNDynamics
    from ml_conformer_generator.src.mlconfgen.equivariant_diffusion import PredefinedNoiseSchedule
    from ml_conformer_generator.src.mlconfgen.equivariant_flow_matching import EquivariantFlowMatching
    from ml_conformer_generator.src.mlconfgen.utils import CONTEXT_NORMS, MAX_N_NODES, remove_mean_with_mask
    from ml_conformer_generator.src.mlconfgen.utils.mol_utils import prepare_masks

# --------------------- config ---------------------
device = "cuda"
BATCH = 128
HIDDEN_NF = 420
TEACHER_T = 100
NOISE_PRECISION = 1e-5
LR = 5e-5                 # lower: fine-tuning an already-distilled model
EMA_DECAY = 0.99          # short window; 0.999 was useless at this step count
PAD_TO = MAX_N_NODES
PART_CYCLE = [0, 1]
N = 4
EPOCHS = 8
DT = 1.0 / N

PAIR_DIR = Path("./teacher_pairs/teacher_pairs")
EDM_WEIGHTS = "edm_moi_chembl_15_39.pt"
SRC_CKPT = Path("./checkpoints_fm/best_420_pd_N8.pt")
CKPT_DIR = Path("./checkpoints_fm")
LOG_PATH = CKPT_DIR / f"train_{HIDDEN_NF}_pd.log"


# --------------------- boilerplate ---------------------
class EMA:
    def __init__(self, beta):
        self.beta = beta

    def update_model_average(self, ma, cur):
        for c, m in zip(cur.parameters(), ma.parameters()):
            m.data = m.data * self.beta + (1 - self.beta) * c.data


class Queue:
    def __init__(self, max_len=50):
        self.items, self.max_len = [], max_len

    def __len__(self):
        return len(self.items)

    def add(self, x):
        self.items.insert(0, x)
        if len(self) > self.max_len:
            self.items.pop()

    def mean(self):
        return float(np.mean(self.items))

    def std(self):
        return float(np.std(self.items))


def gradient_clipping(model, q):
    mx = 1.5 * q.mean() + 2.0 * q.std()
    gn = float(torch.nn.utils.clip_grad_norm_(model.parameters(), mx))
    q.add(mx if gn > mx else gn)


def log(msg):
    line = f"{datetime.now().isoformat(timespec='seconds')}  {msg}"
    print(line)
    with open(LOG_PATH, "a") as f:
        f.write(line + "\n")


def com_project(z, node_mask):
    return torch.cat(
        [remove_mean_with_mask(z[..., :3], node_mask), z[..., 3:]], -1
    ) * node_mask


def load_part_pairs(part_i):
    paths = sorted(PAIR_DIR.glob(f"part_{part_i + 1}_shard*.pt")) or sorted(
        PAIR_DIR.glob(f"part{part_i + 1}_shard*.pt")
    )
    if not paths:
        paths = sorted(PAIR_DIR.glob("shard*.pt"))
        if part_i != 0:
            raise FileNotFoundError(PAIR_DIR)
    packs = [torch.load(p, map_location="cpu", weights_only=False) for p in paths]
    return (
        torch.cat([p["x1"] for p in packs]),
        torch.cat([p["n_atoms"] for p in packs]),
        torch.cat([p["context"] for p in packs]),
    )


def alpha_sigma(t, sched, like):
    T = sched.timesteps
    u = (t * T).clamp(0, T)
    i0 = u.floor().long().clamp(0, T - 1)
    frac = (u - i0.float()).clamp(0.0, 1.0)
    g = sched.gamma
    gamma = g[i0] + frac * (g[i0 + 1] - g[i0])
    a = torch.sqrt(torch.sigmoid(-gamma)).view(like.shape[0], 1, 1)
    s = torch.sqrt(torch.sigmoid(gamma)).view(like.shape[0], 1, 1)
    return a, s


def x_eps_from_v(z, v, a, s):
    return a * z - s * v, s * z + a * v


def ddim_v(z, v, a_t, s_t, a_s, s_s, node_mask):
    x, eps = x_eps_from_v(z, v, a_t, s_t)
    return com_project(a_s * x + s_s * eps, node_mask)


def pd_x_target(z_t, z_s_tilde, a_t, s_t, a_s, s_s):
    ratio = s_s / s_t.clamp_min(1e-4)
    denom = (a_s - a_t * ratio).clamp_min(1e-4)
    return (z_s_tilde - ratio * z_t) / denom


def snr_weight(a, s):
    return torch.clamp(a * a / (s * s).clamp_min(1e-8), min=1.0)


def x_loss(x_pred, x_tgt, w, node_mask):
    err = (x_pred - x_tgt) ** 2 * node_mask * w
    n = node_mask.sum().clamp_min(1)
    lx = err[..., :3].sum() / (n * 3)
    lh = err[..., 3:].sum() / (n * 8)
    return lx + lh, lx.detach(), lh.detach()


# --------------------- models (from best_420_pd_N8) ---------------------
edm_ckpt = torch.load(EDM_WEIGHTS, map_location="cpu", weights_only=False)
norms = {
    k: torch.tensor(v, device=device, dtype=torch.float32)
    for k, v in edm_ckpt.get("context_norms", CONTEXT_NORMS).items()
}
sched = PredefinedNoiseSchedule(timesteps=TEACHER_T, precision=NOISE_PRECISION).to(device)

src = torch.load(SRC_CKPT, map_location=device, weights_only=False)
assert src.get("param") == "v" and int(src["hidden_nf"]) == HIDDEN_NF


def make_model(state):
    m = EquivariantFlowMatching(
        dynamics=EGNNDynamics(
            in_node_nf=9, context_node_nf=3, hidden_nf=HIDDEN_NF, device=device
        ),
        in_node_nf=8,
    ).to(device)
    m.load_state_dict(state)
    return m


# non-EMA weights: visually better at 8 NFE, and EMA lagged at this step count
jump_teacher = make_model(src["student"]).eval()
for p in jump_teacher.parameters():
    p.requires_grad_(False)

student = make_model(src["student"])
student_ema = copy.deepcopy(student).to(device)
for p in student_ema.parameters():
    p.requires_grad_(False)
ema = EMA(EMA_DECAY)
opt = AdamW(student.parameters(), lr=LR, amsgrad=True, weight_decay=1e-12)
q = Queue()
q.add(3000.0)


# --------------------- N=4 halving ---------------------
def stage_c_step(x1_b, nm, em, bctx):
    B, Nn = x1_b.shape[:2]
    i = torch.randint(1, N + 1, (B, 1), device=device).float()
    t = i * DT
    s = t - DT
    tm = t - 0.5 * DT
    eps = student.sample_combined_position_feature_noise(B, Nn, nm)
    a_t, s_t = alpha_sigma(t, sched, x1_b)
    a_m, s_m = alpha_sigma(tm, sched, x1_b)
    a_s, s_s = alpha_sigma(s, sched, x1_b)
    zt = com_project(a_t * x1_b + s_t * eps, nm)
    with torch.no_grad():
        v1 = jump_teacher.velocity(zt, t, nm, em, bctx)
        zm = ddim_v(zt, v1, a_t, s_t, a_m, s_m, nm)
        v2 = jump_teacher.velocity(zm, tm, nm, em, bctx)
        zs = ddim_v(zm, v2, a_m, s_m, a_s, s_s, nm)
        x_tgt = com_project(pd_x_target(zt, zs, a_t, s_t, a_s, s_s), nm)
    v_pred = student.velocity(zt, t, nm, em, bctx)
    x_pred, _ = x_eps_from_v(zt, v_pred, a_t, s_t)
    return x_loss(x_pred, x_tgt, snr_weight(a_t, s_t), nm)


log(f"=== Stage C: N={N} (dt={DT:.4f}) from {SRC_CKPT.name}, non-EMA jump teacher, ema={EMA_DECAY} ===")
best = float("inf")
for epoch in range(EPOCHS):
    part_i = PART_CYCLE[epoch % len(PART_CYCLE)]
    x1_all, na_all, ctx_all = load_part_pairs(part_i)
    perm = torch.randperm(x1_all.shape[0])
    x1_all, na_all, ctx_all = x1_all[perm], na_all[perm], ctx_all[perm]
    student.train()
    run, rx, rh, ns = 0.0, 0.0, 0.0, 0
    pbar = tqdm(range(0, x1_all.shape[0] - BATCH + 1, BATCH), desc=f"N{N} ep{epoch}")
    for i in pbar:
        sl = slice(i, i + BATCH)
        na = na_all[sl].to(device=device, dtype=torch.long)
        nm, em = prepare_masks(na, PAD_TO, device)
        ctx = ctx_all[sl].to(device=device, dtype=torch.float32)
        bctx = ((ctx - norms["mean"]) / norms["mad"]).unsqueeze(1).expand(-1, PAD_TO, -1) * nm
        x1_b = com_project(x1_all[sl].to(device=device, dtype=torch.float32), nm)
        loss, lx, lh = stage_c_step(x1_b, nm, em, bctx)
        if not torch.isfinite(loss):
            log(f"WARN non-finite ep={epoch}")
            continue
        opt.zero_grad(set_to_none=True)
        loss.backward()
        gradient_clipping(student, q)
        opt.step()
        ema.update_model_average(student_ema, student)
        run += loss.item(); rx += lx.item(); rh += lh.item(); ns += 1
        if ns % 20 == 0:
            pbar.set_postfix(loss=f"{run/ns:.4f}", lx=f"{rx/ns:.4f}", lh=f"{rh/ns:.4f}")
    avg = run / max(ns, 1)
    log(f"N{N} epoch={epoch} avg={avg:.6f} lx={rx/max(ns,1):.6f} lh={rh/max(ns,1):.6f}")
    ckpt = {
        "epoch": epoch, "avg_loss": avg, "stage": f"N{N}",
        "nfe": N, "param": "v", "hidden_nf": HIDDEN_NF,
        "student": student.state_dict(), "student_ema": student_ema.state_dict(),
        "opt": opt.state_dict(),
    }
    torch.save(ckpt, CKPT_DIR / f"latest_{HIDDEN_NF}_pd.pt")
    if avg < best:
        best = avg
        torch.save(ckpt, CKPT_DIR / f"best_{HIDDEN_NF}_pd_N{N}.pt")
        log(f"ckpt best N{N} avg={avg:.6f}")

log(f"done N={N}. Gate best_{HIDDEN_NF}_pd_N{N}.pt with 4-step v-DDIM; compare student vs student_ema.")